<table  align="left" width="100%"> <tr>
        <td  style="background-color:#ffffff;"><a href="https://qworld.net" target="_blank"><img src="../images/qworld/qworld.jpg" width="35%" align="left"></a></td>
        <td  align="right" style="background-color:#ffffff;vertical-align:bottom;horizontal-align:right">
            prepared by Özlem Salehi Köken
        </td>        
</tr></table>

# Variational Quantum Eigensolver - Implementation



In the previous notebook, we learnt about the variational principle, which lies at the heart of VQE. In this notebook, we will implement VQE which relies on this principle.

## How to implement VQE?

VQE works by iteratively preparing a trial quantum state $|\psi(\theta) \rangle$ with adjustable parameters, estimating $\langle \psi(\theta)| H | \psi(\theta)\rangle$ on a quantum computer, and using classical optimization to refine the parameters $\theta$ until the lowest possible energy (ground state) is approximated.

In other words, VQE allows practical estimation of $\langle \psi(\theta) | H |  \psi(\theta) \rangle$ for different trial states $\ket{\psi (\theta) }$. 

### Ansatz

"Trial quantum state $|\psi(\theta) \rangle$ with adjustable parameters" is the ansatz we discussed in the previous notebook. The quality of the ansatz directly affects the efficiency of the algorithm—too simple, and it may not capture the correct solution; too complex, and it may be difficult to optimize.

Common approaches include:

- Hardware-Efficient Ansatz: Uses native quantum gates but may suffer from barren plateaus.
- Unitary Coupled Cluster (UCC) Ansatz: Popular in quantum chemistry but requires deep circuits.
- Problem-Specific Ansatz: Tailored to a given Hamiltonian, often improving efficiency.
- Adaptive Ansatz (e.g., ADAPT-VQE): Dynamically constructs an optimal ansatz, reducing circuit depth.

In this notebook, we will not go into details of the mentioned ansatze and keep things simple.

The next thing we need to understand is how VQE "estimates $\langle \psi(\theta)| H | \psi(\theta)\rangle$ on a quantum computer".


### What is $\langle \psi(\theta)| H | \psi(\theta)\rangle$ ?

The quantity $\langle \psi(\theta)| H | \psi(\theta)\rangle$ is called the expectation value of $H$ with respect to state $|\psi(\theta)\rangle$. Let's understand why this is the case before answering the question how to estimate this using a quantum computer.

The general way to define a measurement in quantum mechanics is based on the concept of observable operators. An observable in quantum mechanics (such as position, momentum, or spin) is represented by a Hermitian operator. This is because measurements in quantum mechanics must produce real eigenvalues, which are physically interpretable as the possible outcomes of a measurement. Any such observable can be expressed as

$$
H = \sum_\lambda \lambda P_\lambda,
$$
where $P_\lambda$ is the projector onto the eigenspace of $H$ with eigenvalue $\lambda$. The projectors are constructed as $P_\lambda = \ket{\psi_\lambda} \bra{\psi_\lambda}$, where $\ket{\psi_\lambda}$ are the normalized eigenvectors of $H$, which is always possible to find since the observable is Hermitian.



### Task 1

For $H=Z$, compute $P_\lambda$.

[click for our solution](04_variational_quantum_eigensolver_implementation_solutions.ipynb#task1)

---

Now let's replace $\sum_\lambda \lambda P_\lambda$ for $H$ in $ \bra{\psi} H\ket{\psi}$.

$ \bra{\psi} H\ket{\psi} = \bra{\psi} \sum_\lambda \lambda P_\lambda\ket{\psi} =  \sum_\lambda  \lambda \bra{\psi} P_\lambda\ket{\psi} = \sum_\lambda  \lambda  \bra{\psi} \ket{\psi_\lambda} \bra{\psi_\lambda}\ket{\psi}  =  \sum_\lambda  \lambda  |\bra{\psi_\lambda}{\psi} \rangle|^2  = \sum_\lambda  \lambda |c_\lambda|^2 $

Let's inspect the quantity $|\bra{\psi_\lambda}{\psi} \rangle|^2$

When a measurement is performed, the system collapses to one of the eigenstates of the operator and the outcome of the measurement corresponds to the eigenvalue associated with that eigenstate, with the following probability:

$$
p(\lambda) = |\bra{\psi_\lambda}{\psi}\rangle|^2 = |c_\lambda|^2
$$

This is known as the Born Rule. The state of the quantum system after the measurement is given by

$$
\frac{P_\lambda \ket{\psi}}{\sqrt {p(\lambda)}}.
$$




Hence we can conclude that 

$$
\bra{\psi}H\ket{\psi} =  \sum_\lambda  \lambda \bra{\psi} P_\lambda\ket{\psi}  = \sum_\lambda  \lambda p(\lambda) = E(\lambda).
$$

Recall the definition of expectation value from probability theory, which is simply a summation over probabilities and outcomes. With this observation, we can conclude that $\bra{\psi}H\ket{\psi}$ is the expectation value of operator $H$.

### Task 2

Compute the expectation value of $Z$ operator with respect to the state $\ket{\psi} = \frac{1}{\sqrt 3} \ket 0 + \frac{\sqrt 2}{\sqrt 3} \ket 1$. Use the formula $
\bra{\psi}H\ket{\psi} =  \sum_\lambda  \lambda \bra{\psi} P_\lambda\ket{\psi}  = \sum_\lambda  \lambda p(\lambda) = E(H)$.

Also compute $\bra{\psi}H\ket{\psi}$ directly and compare.

[click for our solution](04_variational_quantum_eigensolver_implementation_solutions.ipynb#task2)

### How to estimate expectation values using a quantum circuit?

Instead of specifying an observable, the phrase "measurement in computational basis" is often used, where the basis is formed by the corresponding eigenstates of some observable. In quantum computing, in general we measure in the computational basis $\{\ket{0}, \ket{1}\}$, which actually corresponds to measuring with observable $Z$. 

Suppose that we prepare the quantum state $\psi$ using a quantum circuit and we measure the circuit many times. By the measurement outcomes, we can make an estimation on the expectation value of the $Z$ operator, by estimating the probabilities $p(\lambda)$.

Note that when we measure $\ket{1}$, the corresponding eigenvalue is -1 and when we measure $\ket{0}$, the corresponding eigenvalue is 1.



### Task 3

Estimate the expectation value of $Z$ with respect to the state $\ket{\psi} = \frac{1}{\sqrt 3} \ket 0 + \frac{\sqrt 2}{\sqrt 3} \ket 1$ by taking 10, 100, and 1000 measurements. 

Note: To initialize your circuit with the state $\ket{\psi} = \frac{1}{\sqrt 3} \ket 0 + \frac{\sqrt 2}{\sqrt 3} \ket 1$, you can use the following:





In [ ]:
from qiskit.circuit.library import StatePreparation
import math
from qiskit import QuantumCircuit
from qiskit import transpile

stateprep = StatePreparation([1/math.sqrt(3), math.sqrt(2)/math.sqrt(3)])
qc = QuantumCircuit(1)
qc.append(stateprep, [0]) # Apply the state preparation circuit to qubit 0
qc = transpile(qc, basis_gates=['u', 'cx']) # Transpile the circuit to use only 'u' and 'cx' gates

In [ ]:
from qiskit import  QuantumCircuit, transpile
from qiskit.circuit.library import StatePreparation
from qiskit_aer import AerSimulator
import math

# Your code here

for shots in [10, 100, 1000]:


    job = AerSimulator().run(qc,shots=shots)
    counts = job.result().get_counts(qc)
    e = (-1*counts.get("1", 0)/ shots + 1*counts.get("0", 0) / shots)
    print(counts, "Expectation value of Z is estimated as: ", e)

[click for our solution](04_variational_quantum_eigensolver_implementation_solutions.ipynb#task3)

### How to take expectation value of operators other than $Z$?

Let us consider $X$ operator for instance.

\begin{align*}
\bra{\psi} X\ket{\psi}  = \bra{\psi} HZH \ket{\psi} = \bra{\psi'} Z \ket{\psi'},
\end{align*}

where $\ket{\psi'} = H\ket{\psi}$. This observation suggests us a method for measurement in $X$ basis: We can apply $H$ on the quantum state, we would like to measure and then perform usual measurement with the $Z$ observable. We can make a similar observation for $Y$ operator:

\begin{align*}
\bra{\psi} Y\ket{\psi}  = \bra{\psi} SHZHS^\dagger \ket{\psi} = \bra{\psi''} Z \ket{\psi''},
\end{align*}

where $\ket{\psi''} = HS^\dagger\ket{\psi}$. Hence, measuring with observable $Y$, we can first apply $S^\dagger$ followed by $H$, then measure in the $Z$ basis.


### Task 4

Estimate the expectation value of operator $X$ with respect to the state $\ket{\psi} = \frac{1}{\sqrt 3} \ket 0 + \frac{\sqrt 2}{\sqrt 3} \ket 1$ by taking 1000 measurements. 

In [ ]:
from qiskit import  QuantumCircuit, transpile
from qiskit.circuit.library import StatePreparation
from qiskit_aer import AerSimulator
import math

# Your code here

print(counts, "Expectation value of X is estimated as: ", e)

[click for our solution](04_variational_quantum_eigensolver_implementation_solutions.ipynb#task4)

### Measurement in VQE

In general, the Hamiltonian whose ground state energy we would like to estimate will consist of the composition of Pauli strings i.e.,
$$H = H_1 + H_2 + \cdots + H_n.$$
So to compute $\bra{\psi}H\ket{\psi}$, we can compute individual terms $\bra{\psi}H_i\ket{\psi}$ and take the sum.

Consider $H = 2Y + Z - X$. The Hamiltonian consists of 3 terms, $H_1 = 2Y$, $H_ 2= Z$, and $H_3 = -X$. To compute the expectation value of $|\psi\rangle$ with respect to $H$, we need to prepare 3 different circuits.  



### Task 5

Estimate $\bra{\psi}H\ket{\psi}$ for $\ket{\psi} = \frac{1}{\sqrt 3} \ket 0 + \frac{\sqrt 2}{\sqrt 3} \ket 1$ and $H = 2Y + Z - X$, by estimating $\bra{\psi}H_1\ket{\psi}$, $\bra{\psi}H_2\ket{\psi}$ and $\bra{\psi}H_3\ket{\psi}$ separately and taking the sum.

In [ ]:
from qiskit import  QuantumCircuit, transpile
from qiskit.circuit.library import StatePreparation
from qiskit_aer import AerSimulator
import math
stateprep = StatePreparation([1/math.sqrt(3), math.sqrt(2)/math.sqrt(3)])
shots = 10000

In [ ]:
# Circuit for H1:

qc = QuantumCircuit(1,1)
qc.append(stateprep, [0])

### Your code here

qc.measure(0,0)
qc = transpile(qc, basis_gates=['u', 'cx'])


job = AerSimulator().run(qc,shots=shots)
counts = job.result().get_counts(qc)
e1 = 2*(-1*counts.get("1", 0) / shots + 1*counts.get("0", 0) / shots)
print(counts, "Expectation value of H_1 is estimated as: ", e1)

In [ ]:
qc = QuantumCircuit(1,1)
qc.append(stateprep, [0])

### Your code here


job = AerSimulator().run(qc,shots=shots)
counts = job.result().get_counts(qc)
e2 = (-1*counts.get("1", 0) / shots + 1*counts.get("0", 0) / shots)
print(counts, "Expectation value of H_2 is estimated as: ", e2)

In [ ]:
qc = QuantumCircuit(1,1)
qc.append(stateprep, [0])

### Your code here

qc.measure(0,0)
qc = transpile(qc, basis_gates=['u', 'cx'])


job = AerSimulator().run(qc,shots=shots)
counts = job.result().get_counts(qc)
e3 = -1*(-1*counts.get("1", 0) / shots + 1*counts.get("0", 0) / shots)
print(counts, "Expectation value of H_3 is estimated as: ", e3)

In [ ]:
print("Estimated expectation value for H: ", e1+e2+e3)

[click for our solution](04_variational_quantum_eigensolver_implementation_solutions.ipynb#task5)

### Task 6

Find exactly the expectation value of $H = 2Y + Z - X$ with respect to the state $\ket{\psi} = \frac{1}{\sqrt 3} \ket 0 + \frac{\sqrt 2}{\sqrt 3} \ket 1$ using `expectation_value` function of `Statevector` and compare the results with Task 5.

In [ ]:
from qiskit.quantum_info import Statevector
from qiskit.quantum_info import SparsePauliOp


### Your code here


print("Expectation value:", expectation_value)

[click for our solution](04_variational_quantum_eigensolver_implementation_solutions.ipynb#task6)

### What to do when we have multiple qubits?

Suppose that our Hamiltonian is given by $Z_0Z_1$. The eigenstates of $Z_0Z_1$ are $\ket{00}$, $\ket{01}$, $\ket{10}$, $\ket{11}$, with the corresponding eigenvalues $1, -1, -1, 1$ respectively. How can we implement this behaviour?

Note that eigenvalues are 1 if two qubits are in the same state, and -1 otherwise.

Let us implement a $CNOT$ gate on the two qubits as $CNOT\ket{q_0}\ket{q_1} =\ket{q_0}\ket{q_0 \oplus q_1} $,  
- $\ket{q_0 \oplus q_1} = \ket{0}$, if two qubits have the same state
- $\ket{1}$, otherwise.

After applying a CNOT, measurement with $Z$ on qubit $q_1$ yields us the result we desire.


This idea can be extended to multiple qubits by applying CNOT in a chain structure. Such CNOT chain keeps the parity of 1s in the state and results in state $\ket{1}$ on the final target qubit if the number of 1s in the state is odd, and $\ket{0}$ otherwise. Hence, a measurement on the final qubit with observable $Z$ coincides with the eigenvalues of the multiqubit observable.

### Task 7

Create an equal superposition state on two qubits. The expectation value of $Z_0Z_1$ with respect to the equal superposition state is 0. Verify this from the measurement results.  

In [ ]:
from qiskit import  QuantumCircuit, transpile
from qiskit.circuit.library import StatePreparation
from qiskit_aer import AerSimulator

# Create and initialize circuit
### Your code here

# Measurement with Z0Z1
### Your code here

shots = 10000
job = AerSimulator().run(qc,shots=shots)
counts = job.result().get_counts(qc)

e = ### Your code here
print(counts)
print("Expectation value of Z0Z1 is estimated as: ", e)

[click for our solution](04_variational_quantum_eigensolver_implementation_solutions.ipynb#task7)

#### What about other observables like $Y_0X_1Z_2$ ?

Such observables can be obtained from $Z_0Z_1Z_2$, by making the necessary transformation through $H$ and $HS^{\dagger}$.

<img src="../images/vqe_measurement.png" align="center">

### Task 8

Compute the expectation value of $Y_0X_2Z_3$ with respect to quantum state $\ket{0101}$.

In [ ]:
from qiskit import  QuantumCircuit, transpile
from qiskit.circuit.library import StatePreparation
from qiskit_aer import AerSimulator

# Prepare circuit and initial state

# Measurement with Y0X2Z3
### Your code here

shots = 10000
job = AerSimulator().run(qc,shots=shots)
counts = job.result().get_counts(qc)
e = -1*counts.get("1",0) / shots + 1*counts.get("0",0) / shots
print(counts)
print("Expectation value of Y0X2Z3 is estimated as: ", e)

[click for our solution](04_variational_quantum_eigensolver_implementation_solutions.ipynb#task8)

### Number of measurements in VQE

A key challenge in VQE is the number of measurements needed to estimate expectation values accurately.

The required measurement count scales inversely with the desired accuracy.
There are strategies to reduce this, like grouping commuting terms to reduce the number of separate measurements needed.

## The algorithm

Now we can put the pieces together for the VQE algorithm.


<img src="../images/vqe.png" align="center">

- We select a parametrized quantum circuit $\ket{\psi(\theta)}$ to serve as our ansatz.
- We initialize its parameters to $\theta_0$, preparing $\ket{\psi(\theta_0)}$.
- We use a classical optimizer to adjust the parameter $\theta$, by performing measurements to compute $\bra{\psi(\theta_0)}H\ket{\psi(\theta_0)}$, which is the cost function we would like to minimize.

VQE is a hybrid algorithm, where the optimization of the parameters is performed on a classical computer, and the quantum computer is used for calculating the expectation value.

### Task 9

Using the ansatz given below, run VQE to estimate the ground state energy of the Hamiltonian $H = 2Y_0Y_1 + Z_0X_1 - X_0Z_1$. Compute the ground state energy exactly to verify your findings.



In [ ]:
from qiskit import  QuantumCircuit
from qiskit.circuit import Parameter

def ansatz():
    qc = QuantumCircuit(2,1)
    theta_0 = Parameter('theta_0')
    theta_1 = Parameter('theta_1')
    qc.rx(theta_0,0)
    qc.cx(0,1)
    qc.ry(theta_1,1)
    return qc

We divide this task into smaller steps for you.

To start with, find the lowest eigenvalue of our operator to verify your findings later on.


In [ ]:
import numpy as np
from qiskit.quantum_info import SparsePauliOp

# Define the Hamiltonian H

# Calculate eigenvalues of H

print("Eigenvalues:", eigenvalues)
print("Ground state energy (smallest eigenvalue):", np.min(eigenvalues))

Complete the following function, that returns the necessary quantum circuit for measurement with respect to the given Hamiltonian `op`.

In [ ]:
# op will be one of Y0Y1, Z0X1, X0Z1
def measurement_circuit(op):
    qc = QuantumCircuit(2)
    ### Your code here 

    #Check the indices of the Pauli operators and apply the necessary gates to measure in the correct basis)


    return qc

Complete the function below, that computes the expectation value of the ansatz with respect to $H$, based on the given parameters.

In [ ]:
from qiskit.circuit import Parameter
from qiskit import  QuantumCircuit, transpile
from qiskit.circuit.library import StatePreparation
from qiskit_aer import AerSimulator

#The first parameters should be always the parameters
def expectation_value(params, H):
    e = 0 # Initialize expectation value
    for op in H:
    ### Your code here
        #Construct the ansatz

        #Assign the parameters

        #Add the measurement circuit and measure

        #Transpile the circuit for convenience
        
        #Run the circuit and get the counts
        
        #Compute expectation for each term in the Hamiltonian and sum them up

    #The function should return a real number
    #Even very small imaginary parts can cause optimizer to throw an error
    return e.real

We set some initial values and check the function's value at the initial parameters.

In [ ]:
# Set initial parameters

e = expectation_value(params, H)
print("Initial expectation value estimation:", e)

Use a classical optimizer to find the parameters, which minimizes the function.


In [ ]:
from scipy.optimize import minimize

#Note that for x0 we pass some initial parameters 
#and for args we pass the second argument of the expectation value function which is the Hamiltonian H
result = minimize(expectation_value, x0 = params, args=(H,), method='COBYLA', options = {"maxiter" : 10000})
print("Optimized parameters:", result.x)
print("Expectation value at optimized parameters:", expectation_value(result.x, H))

[click for our solution](04_variational_quantum_eigensolver_implementation_solutions.ipynb#task9)

### Task 10

`Efficient SU2` circuit consists of layers of single-qubit operations spanned by SU(2) and CNOT gates. SU(2) stands for the special unitary group of degree 2, whose elements are
$2 \times 2$ unitary matrices with determinant 1.

Below is such ansatz on 2 qubits.


<img src="../images/su2.jpeg" align="center">

In [ ]:
from qiskit.circuit.library import EfficientSU2

def ansatz2():
    qc = QuantumCircuit(2,1)
    qc.compose(EfficientSU2(2), inplace=True)
    return qc

Repeat Task 9 with the SU2 ansatz.

[click for our solution](04_variational_quantum_eigensolver_implementation_solutions.ipynb#task10)

## What else about VQE?

VQE is suitable for NISQ (Noisy Intermediate-Scale Quantum) devices due to several reasons.

- Near-Term Applicability: Unlike other quantum algorithms requiring full fault tolerance, VQE can run on noisy intermediate-scale quantum (NISQ) devices.

- Variational Flexibility: The algorithm allows for flexible ansatz choices, enabling the adaptation of quantum circuits to hardware constraints.

- Hybrid Nature: By offloading optimization tasks to classical computers, VQE mitigates some quantum computational limitations.

- Error Mitigation Techniques: VQE can incorporate classical and quantum error mitigation strategies, improving accuracy on noisy devices.

Some Disadvantages of VQE:

- Measurement Overhead: Computing expectation values requires a large number of measurements, making VQE expensive in terms of runtime.
- Optimization Challenges: Classical optimization can struggle due to barren plateaus (where gradients vanish), leading to inefficient training.
- Ansatz Design Trade-offs: While expressive ansatz can capture complex wavefunctions, they often require deep circuits, which are difficult to implement on NISQ hardware.
